# Build the static sector adjacency matrix

Produces `data/sectors_static.parquet` — a 98×98 binary matrix where `A[i,j] = 1` iff stocks `i` and `j` belong to the same GICS sector. This is the *baseline graph* for the `static_graph` ablation: a hand-coded, time-invariant relation graph that the learned `GraphConstructor` is supposed to beat.

Sector source: yfinance `Ticker.info['sector']` (free, public). Fallback for missing sectors: GICS labels hard-coded below.

Conventions:
- Symmetric (`A == A.T`).
- Self-loops zeroed (`diag(A) = 0`); GAT re-adds them.
- No row-normalisation — the GAT does its own softmax.
- Same ticker ordering as `data/stocks_dataset.parquet` (alphabetical).

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

REPO = Path.cwd()
if REPO.name in ('data-preprocessing', 'notebooks'):
    REPO = REPO.parent

# Read the canonical ticker list from the existing dataset so ordering matches.
STOCKS_PARQUET = REPO / 'data' / 'stocks_dataset.parquet'
OUT_PATH       = REPO / 'data' / 'sectors_static.parquet'

tickers_df = pd.read_parquet(STOCKS_PARQUET, columns=['label_up'])
TICKERS = sorted(tickers_df.index.get_level_values('ticker').unique().tolist())
print(f'universe: {len(TICKERS)} tickers')

## 1. Fetch sector per ticker

Try `yfinance` first; fall back to the hand-coded GICS table below. yfinance occasionally returns `None` / `'Unknown'` / blank — caught by the fallback.

In [ ]:
# GICS sector hard-coded fallback (S&P-100 universe). Used when yfinance fails
# or returns missing values. Source: SPDR sector ETF constituents (verified Q1 2025).
GICS_FALLBACK = {
    # Information Technology
    'AAPL':'tech','MSFT':'tech','NVDA':'tech','GOOGL':'tech','META':'tech',
    'AVGO':'tech','ORCL':'tech','CRM':'tech','ADBE':'tech','CSCO':'tech',
    'TXN':'tech','IBM':'tech','INTC':'tech','QCOM':'tech','AMD':'tech',
    'V':'tech','MA':'tech',
    # Communication Services
    'NFLX':'comms','CMCSA':'comms','DIS':'comms','T':'comms','VZ':'comms','TMUS':'comms','CHTR':'comms',
    # Consumer Discretionary
    'AMZN':'discretionary','TSLA':'discretionary','HD':'discretionary','MCD':'discretionary',
    'NKE':'discretionary','LOW':'discretionary','SBUX':'discretionary','TGT':'discretionary',
    'BKNG':'discretionary','F':'discretionary','GM':'discretionary',
    # Consumer Staples
    'PG':'staples','KO':'staples','PEP':'staples','COST':'staples','WMT':'staples',
    'MO':'staples','PM':'staples','CL':'staples','MDLZ':'staples',
    # Health Care
    'JNJ':'health','UNH':'health','LLY':'health','PFE':'health','ABBV':'health','MRK':'health',
    'TMO':'health','ABT':'health','BMY':'health','CVS':'health','AMGN':'health','DHR':'health',
    'MDT':'health','GILD':'health','ISRG':'health',
    # Financials
    'JPM':'financials','BAC':'financials','WFC':'financials','GS':'financials','MS':'financials',
    'C':'financials','BLK':'financials','SCHW':'financials','AXP':'financials','BK':'financials',
    'COF':'financials','USB':'financials','MET':'financials','AIG':'financials','BRK-B':'financials',
    'SPG':'financials',
    # Industrials
    'CAT':'industrials','BA':'industrials','HON':'industrials','UPS':'industrials','GE':'industrials',
    'UNP':'industrials','RTX':'industrials','LMT':'industrials','DE':'industrials','MMM':'industrials',
    'FDX':'industrials','GD':'industrials','EMR':'industrials',
    # Energy
    'XOM':'energy','CVX':'energy','COP':'energy','SLB':'energy',
    # Materials
    'LIN':'materials',
    # Utilities
    'NEE':'utilities','SO':'utilities','DUK':'utilities',
    # Real Estate
    'AMT':'real_estate',
}

def normalise_sector(s):
    if not s or str(s).lower() in ('none', 'nan', 'unknown', ''):
        return None
    s = str(s).lower().strip()
    # Roughly map yfinance's 'sector' strings to our GICS keys.
    mapping = {
        'technology': 'tech', 'communication services': 'comms',
        'consumer cyclical': 'discretionary', 'consumer defensive': 'staples',
        'healthcare': 'health', 'financial services': 'financials',
        'industrials': 'industrials', 'energy': 'energy',
        'basic materials': 'materials', 'utilities': 'utilities',
        'real estate': 'real_estate',
    }
    return mapping.get(s, s.replace(' ', '_'))

sectors = {}
for i, t in enumerate(TICKERS):
    s = None
    try:
        s = normalise_sector(yf.Ticker(t).info.get('sector'))
    except Exception:
        s = None
    if s is None:
        s = GICS_FALLBACK.get(t)
    sectors[t] = s
    if (i + 1) % 25 == 0:
        print(f'  fetched {i+1}/{len(TICKERS)}')

missing = [t for t, s in sectors.items() if s is None]
print(f'unmapped tickers: {missing}')
for t in missing:
    sectors[t] = 'other'

sec_series = pd.Series(sectors, name='sector').sort_index()
print('\nsector counts:')
print(sec_series.value_counts())

## 2. Build the binary same-sector adjacency

In [ ]:
N = len(TICKERS)
sec_arr = sec_series.reindex(TICKERS).to_numpy()
A = (sec_arr[:, None] == sec_arr[None, :]).astype(np.float32)
np.fill_diagonal(A, 0.0)

# Sanity: symmetry + non-trivial structure
assert np.allclose(A, A.T), 'matrix is not symmetric'
deg = A.sum(axis=1)
print(f'matrix: {A.shape}   total edges: {int(A.sum()/2)}   density: {A.mean()*100:.2f}%')
print(f'degree per stock — min {int(deg.min())}, mean {deg.mean():.1f}, max {int(deg.max())}')
isolated = [TICKERS[i] for i in range(N) if deg[i] == 0]
if isolated:
    print(f'isolated tickers (no same-sector peer): {isolated}')
else:
    print('no isolated tickers ✓')

## 3. Save

In [ ]:
df = pd.DataFrame(A, index=TICKERS, columns=TICKERS)
df.to_parquet(OUT_PATH, compression='zstd')
print(f'wrote {OUT_PATH}   ({OUT_PATH.stat().st_size/1024:.1f} KB)')

# Also save the sector mapping as a sidecar CSV — handy for inspection / paper tables.
sec_csv = OUT_PATH.with_suffix('.csv').with_name('sectors_static_labels.csv')
sec_series.reset_index().rename(columns={'index': 'ticker'}).to_csv(sec_csv, index=False)
print(f'wrote {sec_csv}')